In [1]:
# WHAT WE SHOULD BUILD ⭐⭐⭐

# We should engineer features across:

# Category	Meaning
# Affordability	repayment capacity
# Leverage	debt stress
# Behavioral	repayment consistency
# Intensity	borrowing aggressiveness
# Stability	financial resilience
# Temporal	repayment deterioration

In [2]:
# IMPORTS
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    100
)

In [3]:
# LOAD DATASETS (https://www.kaggle.com/competitions/home-credit-default-risk/data)

# Used by the bank to decide:

# Should we approve this customer?
# IMPORTANT COLUMNS
# Column	Meaning
# SK_ID_CURR	customer id
# TARGET	default outcome
# AMT_CREDIT	requested loan amount
# AMT_INCOME_TOTAL	customer income
# NAME_CONTRACT_TYPE	loan type
# DAYS_EMPLOYED	employment history
# EXT_SOURCE_*	external risk scores
application_train = pd.read_csv(
    "../data/raw/application_train.csv"
)


# application_test = pd.read_csv(
#     "../data/raw/application_test.csv"
# )

# BUSINESS MEANING

# The bank is checking:

# What loans does this customer
# already have with OTHER banks?

# Exactly like real banking underwriting.

# ONE CUSTOMER CAN HAVE MANY BUREAU RECORDS

# Meaning:

# 1 customer
#     ↓
# many external credits

# This is why:

# bureau.shape

# is HUGE.

# IMPORTANT COLUMNS
# Column	Meaning
# SK_ID_CURR	customer id
# SK_ID_BUREAU	bureau credit id
# CREDIT_ACTIVE	active/closed loan
# AMT_CREDIT_SUM	external debt
# AMT_CREDIT_SUM_DEBT	outstanding debt
# DAYS_CREDIT	how old the credit is
# CREDIT_DAY_OVERDUE	overdue exposure
bureau = pd.read_csv(
    "../data/raw/bureau.csv"
)

# bureau_balance = pd.read_csv(
#     "../data/raw/bureau_balance.csv"
# )

# BUSINESS MEANING

# The bank asks:

# Has this customer applied before?

# And:

# were they approved?
# rejected?
# cancelled?
# how much did they request?
# how risky were they historically?
# ONE CUSTOMER CAN HAVE MANY APPLICATIONS

# Meaning:

# 1 customer
#     ↓
# many historical applications

# This is:

# behavioral credit intelligence.
# IMPORTANT COLUMNS
# Column	Meaning
# SK_ID_CURR	customer id
# SK_ID_PREV	previous application id
# NAME_CONTRACT_STATUS	approved/rejected
# AMT_APPLICATION	requested amount
# AMT_CREDIT	approved amount
# DAYS_DECISION	when decision occurred
# CNT_PAYMENT	installment count
previous_application = pd.read_csv(
    "../data/raw/previous_application.csv"
)

# installments_payments = pd.read_csv(
#     "../data/raw/installments_payments.csv"
# )

# credit_card_balance = pd.read_csv(
#     "../data/raw/credit_card_balance.csv"
# )

# pos_cash_balance = pd.read_csv(
#     "../data/raw/POS_CASH_balance.csv"
# )

In [4]:
installments_payments = pd.read_csv(
    "../data/raw/installments_payments.csv"
)

In [ ]:
# SECTION 1 — AFFORDABILITY FEATURES ⭐⭐⭐


In [5]:
# 1. Annuity-to-Income Ratio

# ------------------------------------------------------------
# ANNUITY_TO_INCOME
# ------------------------------------------------------------
# Measures repayment burden relative to income.
#
# Business Meaning:
# Higher values may indicate affordability stress
# and reduced financial flexibility.
#
# Formula:
# Monthly Loan Repayment / Annual Income
#
# Enterprise Use:
# Used in affordability assessment,
# underwriting, and probability of default models.


application_train[
    "ANNUITY_TO_INCOME"
] = (

    application_train[
        "AMT_ANNUITY"
    ]

    /

    application_train[
        "AMT_INCOME_TOTAL"
    ]
)

/tmp/ipykernel_1520/569885503.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [6]:
# 2. Credit-to-Income Ratio

# ------------------------------------------------------------
# CREDIT_TO_INCOME
# ------------------------------------------------------------
# Measures total requested credit relative
# to customer income capacity.
#
# Business Meaning:
# Indicates leverage intensity and borrowing pressure.
#
# Formula:
# Total Credit Amount / Annual Income
#
# Enterprise Use:
# Commonly used in retail banking
# risk segmentation and lending policies.



application_train[
    "CREDIT_TO_INCOME"
] = (

    application_train[
        "AMT_CREDIT"
    ]

    /

    application_train[
        "AMT_INCOME_TOTAL"
    ]
)

/tmp/ipykernel_1520/1903524854.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [7]:
# 3. Free Cash Flow Estimate ⭐⭐⭐

# ------------------------------------------------------------
# FREE_CASH_FLOW
# ------------------------------------------------------------
# Estimates remaining customer funds after
# repayment obligations.
#
# Business Meaning:
# Higher remaining disposable income may indicate
# stronger repayment resilience.
#
# Formula:
# Annual Income - Loan Annuity
#
# Enterprise Use:
# Used in affordability analysis and
# customer financial stress modeling.


application_train[
    "FREE_CASH_FLOW"
] = (

    application_train[
        "AMT_INCOME_TOTAL"
    ]

    -

    application_train[
        "AMT_ANNUITY"
    ]
)

/tmp/ipykernel_1520/2737661017.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [ ]:
# SECTION 2 — LEVERAGE FEATURES ⭐⭐⭐

# Using bureau data.

In [8]:
# 4. External Debt-to-Income Ratio
# ------------------------------------------------------------
# DEBT_TO_INCOME
# ------------------------------------------------------------
# Measures debt exposure relative to
# customer income capacity.
#
# Business Meaning:
# High values may indicate over-leverage
# and affordability pressure.
#
# Formula:
# External Debt / Annual Income
#
# Enterprise Use:
# One of the most important features in
# enterprise credit risk systems.

customer_external_debt = (

    bureau.groupby(
        "SK_ID_CURR"
    )[
        "AMT_CREDIT_SUM_DEBT"
    ]
    .sum()
    .reset_index()
)

customer_external_debt.columns = [

    "SK_ID_CURR",

    "TOTAL_EXTERNAL_DEBT"
]

application_train = (

    application_train.merge(

        customer_external_debt,

        on="SK_ID_CURR",

        how="left"
    )
)


In [9]:
# After merging debt totals:

application_train[
    "DEBT_TO_INCOME"
] = (

    application_train[
        "TOTAL_EXTERNAL_DEBT"
    ]

    /

    application_train[
        "AMT_INCOME_TOTAL"
    ]
)

/tmp/ipykernel_1520/3072138119.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [10]:
# 5. Debt Per Credit Relationship ⭐⭐⭐

# ------------------------------------------------------------
# DEBT_PER_BUREAU_RECORD
# ------------------------------------------------------------
# Measures average debt concentration across
# bureau credit relationships.
#
# Business Meaning:
# High values may indicate concentrated
# leverage exposure.
#
# Formula:
# Total External Debt / Number of Bureau Records
#
# Enterprise Use:
# Used in concentration risk analysis
# and leverage behavior modeling.

bureau_record_count = (

    bureau.groupby(
        "SK_ID_CURR"
    )
    .size()
    .reset_index(
        name="BUREAU_RECORD_COUNT"
    )
)

application_train = (

    application_train.merge(

        bureau_record_count,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "BUREAU_RECORD_COUNT"
] = (

    application_train[
        "BUREAU_RECORD_COUNT"
    ]
    .fillna(0)
)
# NOW CREATE FEATURE ⭐⭐⭐

# Now this works:

application_train[
    "DEBT_PER_BUREAU_RECORD"
] = (

    application_train[
        "TOTAL_EXTERNAL_DEBT"
    ]

    /

    (
        application_train[
            "BUREAU_RECORD_COUNT"
        ]
        +
        1
    )
)

/tmp/ipykernel_1520/2262301824.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [11]:
# SECTION 3 — BEHAVIORAL FEATURES ⭐⭐⭐

# MOST IMPORTANT SECTION.



In [12]:

# 6. Late Payment Frequency ⭐⭐⭐

# ------------------------------------------------------------
# LATE_PAYMENT_COUNT
# ------------------------------------------------------------
# Number of historically late installment payments.
#
# Business Meaning:
# Measures repayment consistency and
# behavioral repayment stability.
#
# Enterprise Use:
# Strong behavioral feature for
# early warning systems and
# probability of default modeling.
installments_payments[
    "LATE_PAYMENT"
] = (

    installments_payments[
        "DAYS_ENTRY_PAYMENT"
    ]

    >

    installments_payments[
        "DAYS_INSTALMENT"
    ]
)


late_payment_frequency = (

    installments_payments.groupby(
        "SK_ID_CURR"
    )[
        "LATE_PAYMENT"
    ]
    .sum()
    .reset_index()
)

# ------------------------------------------------------------
# STEP 3 — RENAME COLUMN
# ------------------------------------------------------------

late_payment_frequency.columns = [

    "SK_ID_CURR",

    "LATE_PAYMENT_COUNT"
]

# ------------------------------------------------------------
# STEP 4 — MERGE INTO MAIN CUSTOMER DATASET
# ------------------------------------------------------------

application_train = (

    application_train.merge(

        late_payment_frequency,

        on="SK_ID_CURR",

        how="left"
    )
)

# ------------------------------------------------------------
# STEP 5 — HANDLE CUSTOMERS WITH NO LATE PAYMENTS
# ------------------------------------------------------------

application_train[
    "LATE_PAYMENT_COUNT"
] = (

    application_train[
        "LATE_PAYMENT_COUNT"
    ]
    .fillna(0)
)

# ------------------------------------------------------------
# STEP 6 — QUICK VALIDATION
# ------------------------------------------------------------

application_train[
    [
        "SK_ID_CURR",
        "LATE_PAYMENT_COUNT"
    ]
].head()

,SK_ID_CURR,LATE_PAYMENT_COUNT
0,100002,0.0
1,100003,0.0
2,100004,0.0
3,100006,0.0
4,100007,16.0


In [13]:

# 7. Missed Payments Per Loan ⭐⭐⭐

# ------------------------------------------------------------
# MISSED_PAYMENTS_PER_LOAN
# ------------------------------------------------------------
# Measures repayment instability relative
# to historical borrowing activity.
#
# Business Meaning:
# High values indicate repeated repayment
# problems across loans.
#
# Formula:
# Late Payment Count / Previous Loan Count
#
# Enterprise Use:
# Used in behavioral delinquency analysis
# and chronic repayment stress modeling.


previous_application_count = (

    previous_application.groupby(
        "SK_ID_CURR"
    )
    .size()
    .reset_index(
        name="PREVIOUS_APPLICATION_COUNT"
    )
)
application_train = (

    application_train.merge(

        previous_application_count,

        on="SK_ID_CURR",

        how="left"
    )
)
application_train[
    "MISSED_PAYMENTS_PER_LOAN"
] = (

    application_train[
        "LATE_PAYMENT_COUNT"
    ]

    /

    (
        application_train[
            "PREVIOUS_APPLICATION_COUNT"
        ]
        +
        1
    )
)

/tmp/ipykernel_1520/578593020.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [14]:
# 8. Borrowing Intensity ⭐⭐⭐

# ------------------------------------------------------------
# APPLICATIONS_PER_INCOME
# ------------------------------------------------------------
# Measures borrowing intensity relative
# to customer income capacity.
#
# Business Meaning:
# High values may indicate aggressive
# borrowing behavior and elevated
# dependency on external financing.
#
# Customers repeatedly applying for loans
# despite limited income capacity may
# demonstrate increased affordability stress
# and higher repayment vulnerability.
#
# Formula:
# Previous Loan Application Count / Annual Income
#
# Enterprise Use:
# Used in behavioral borrowing analysis,
# credit dependency assessment,
# and customer financial pressure modeling.
#
# This feature helps identify customers
# exhibiting disproportionately high
# borrowing activity relative to
# financial capacity.

application_train[
    "APPLICATIONS_PER_INCOME"
] = (

    application_train[
        "PREVIOUS_APPLICATION_COUNT"
    ]

    /

    application_train[
        "AMT_INCOME_TOTAL"
    ]
)

/tmp/ipykernel_1520/2317217793.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [15]:
# SECTION 4 — TEMPORAL / DELINQUENCY FEATURES ⭐⭐⭐

# VERY important enterprise section.

In [16]:

# 9. Delinquency Severity Index ⭐⭐⭐

# ------------------------------------------------------------
# OVERDUE_PER_BUREAU_RECORD
# ------------------------------------------------------------
# Measures delinquency severity relative
# to the size of the customer's external
# credit ecosystem.
#
# Business Meaning:
# High values may indicate concentrated
# repayment stress across a limited number
# of credit relationships.
#
# This feature contextualizes historical
# delinquency severity against the number
# of bureau-reported credit accounts.
#
# Example:
#
# Customer A:
# 100 overdue days across 50 loans
# -> lower relative stress
#
# Customer B:
# 100 overdue days across 2 loans
# -> concentrated repayment instability
#
# Formula:
# Maximum Historical Overdue Days
# /
# Number of Bureau Credit Records
#
# Enterprise Use:
# Used in contextual delinquency analysis,
# exposure-normalized risk scoring,
# and behavioral repayment stress modeling.
#
# This feature helps distinguish between:
# - broad diversified credit exposure
# - concentrated delinquency behavior
#
# and provides more meaningful risk context
# than overdue severity alone.

customer_max_overdue = (

    bureau.groupby(
        "SK_ID_CURR"
    )[
        "CREDIT_DAY_OVERDUE"
    ]
    .max()
    .reset_index()
)

customer_max_overdue.columns = [

    "SK_ID_CURR",

    "MAX_CREDIT_DAYS_OVERDUE"
]

application_train = (

    application_train.merge(

        customer_max_overdue,

        on="SK_ID_CURR",

        how="left"
    )
)
application_train[
    "OVERDUE_PER_BUREAU_RECORD"
] = (

    application_train[
        "MAX_CREDIT_DAYS_OVERDUE"
    ]

    /

    (
        application_train[
            "BUREAU_RECORD_COUNT"
        ]
        +
        1
    )
)

/tmp/ipykernel_1520/1913076549.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [17]:
# 10. Repayment Stability Score ⭐⭐⭐

# ------------------------------------------------------------
# REPAYMENT_STABILITY
# ------------------------------------------------------------
# Estimates customer repayment consistency
# based on historical late payment behavior.
#
# Business Meaning:
# Higher values indicate more stable
# and reliable repayment behavior.
#
# Customers with fewer late payments
# receive higher stability scores,
# while customers with repeated
# repayment delays receive lower scores.
#
# Formula:
# 1 / (1 + Late Payment Count)
#
# Behavior:
#
# 0 late payments  -> score = 1.0
# 1 late payment   -> score = 0.5
# 5 late payments  -> score = 0.16
# 20 late payments -> score = 0.04
#
# Enterprise Use:
# Used in behavioral stability scoring,
# early warning systems,
# repayment consistency analysis,
# and customer resilience modeling.
#
# This feature transforms raw repayment
# behavior into a normalized stability score
# that can be consumed directly by
# machine learning models.
#
# Lower scores may indicate:
# - chronic repayment instability
# - affordability stress
# - deteriorating financial behavior
# - elevated delinquency risk
#
# Higher scores may indicate:
# - stable repayment discipline
# - strong financial consistency
# - lower operational repayment risk

application_train[
    "REPAYMENT_STABILITY"
] = (

    1

    /

    (
        1
        +
        application_train[
            "LATE_PAYMENT_COUNT"
        ]
    )
)

# VERY interesting engineered feature.

/tmp/ipykernel_1520/875506114.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [ ]:
# ------------------------------------------------------------
# AVG_PAYMENT_DELAY
# ------------------------------------------------------------
# Measures the average lateness severity
# of customer installment payments.
#
# Business Meaning:
# Captures how late customers typically pay
# after scheduled installment deadlines.
#
# Positive delay values indicate:
# late repayment behavior.
#
# Early payments are clipped to zero
# because the analysis focuses specifically
# on repayment delay severity.
#
# Formula:
# Actual Payment Date
# -
# Scheduled Installment Date
#
# Customer-level feature:
# Average PAYMENT_DELAY across all installments.
#
# Example:
#
# Customer A:
# 5 missed payments
# average delay = 1 day
#
# Customer B:
# 5 missed payments
# average delay = 45 days
#
# Although both customers missed the same
# number of payments, Customer B demonstrates
# significantly more severe repayment stress.
#
# Enterprise Use:
# Used in:
# - repayment punctuality analysis
# - behavioral delinquency modeling
# - collections prioritization
# - early warning systems
# - customer financial stress monitoring
#
# This feature provides more granular
# repayment behavior intelligence than
# missed payment frequency alone.
#
# Higher values may indicate:
# - chronic repayment deterioration
# - liquidity pressure
# - affordability stress
# - elevated default vulnerability
#
# Lower values may indicate:
# - mild repayment irregularities
# - operational payment delays
# - relatively stable repayment behavior

installments_payments[
    "PAYMENT_DELAY"
] = (

    installments_payments[
        "DAYS_ENTRY_PAYMENT"
    ]

    -

    installments_payments[
        "DAYS_INSTALMENT"
    ]
)

# ------------------------------------------------------------
# REMOVE EARLY PAYMENTS
# ------------------------------------------------------------
# We focus only on lateness severity.

installments_payments[
    "PAYMENT_DELAY"
] = (

    installments_payments[
        "PAYMENT_DELAY"
    ]
    .clip(lower=0)
)

# ------------------------------------------------------------
# CUSTOMER AVERAGE PAYMENT DELAY
# ------------------------------------------------------------

customer_avg_payment_delay = (

    installments_payments.groupby(
        "SK_ID_CURR"
    )[
        "PAYMENT_DELAY"
    ]
    .mean()
    .reset_index()
)

customer_avg_payment_delay.columns = [

    "SK_ID_CURR",

    "AVG_PAYMENT_DELAY"
]

# ------------------------------------------------------------
# MERGE INTO MAIN DATASET
# ------------------------------------------------------------

application_train = (

    application_train.merge(

        customer_avg_payment_delay,

        on="SK_ID_CURR",

        how="left"
    )
)

# ------------------------------------------------------------
# HANDLE NULLS
# ------------------------------------------------------------

application_train[
    "AVG_PAYMENT_DELAY"
] = (

    application_train[
        "AVG_PAYMENT_DELAY"
    ]
    .fillna(0)
)

# WHY THIS IS STRONGER ⭐⭐⭐

# Compare:

# Customer	Missed Payments	Avg Delay
# A	5	1 day
# B	5	45 days

# Same frequency.
# VERY different risk profile.

# THIS becomes:

# repayment punctuality intelligence.

In [ ]:
# ------------------------------------------------------------
# RECENT_LATE_PAYMENT_COUNT
# ------------------------------------------------------------
# Measures the number of recent late payments
# observed within the customer's repayment history.
#
# Business Meaning:
# Captures active repayment deterioration
# and recent behavioral financial stress.
#
# The feature focuses specifically on
# recent delinquency events because
# recent repayment instability is often
# more predictive than older historical behavior.
#
# Logic:
#
# DAYS_ENTRY_PAYMENT values closer to zero
# represent more recent payment activity.
#
# This feature filters:
# - payments occurring within the most recent
#   90-day behavioral window
# - payments with positive delay severity
#
# Meaning:
# recent late installment behavior.
#
# Formula:
# Count of:
#
# PAYMENT_DELAY > 0
#
# within:
#
# recent 90-day repayment history
#
# Enterprise Interpretation:
#
# Customers with elevated recent late payment
# counts may demonstrate:
#
# - active financial stress
# - worsening repayment behavior
# - affordability deterioration
# - liquidity instability
# - emerging delinquency risk
#
# This feature is significantly more
# operationally relevant than historical
# lifetime aggregates because it reflects
# current behavioral conditions.
#
# Enterprise Use:
# Used heavily in:
#
# - early warning systems
# - collections prioritization
# - behavioral monitoring
# - dynamic probability of default models
# - temporal repayment intelligence systems
#
# Example:
#
# Customer A:
# many historical late payments
# but none recently
#
# -> potentially stabilized customer
#
# Customer B:
# multiple recent late payments
#
# -> active repayment deterioration
#
# This distinction is extremely important
# in enterprise credit risk systems because
# recency often carries stronger predictive
# power than historical severity alone.

recent_late_payments = (

    installments_payments[
        (
            installments_payments[
                "DAYS_ENTRY_PAYMENT"
            ] > -90
        )

        &

        (
            installments_payments[
                "PAYMENT_DELAY"
            ] > 0
        )
    ]
)

recent_late_count = (

    recent_late_payments.groupby(
        "SK_ID_CURR"
    )
    .size()
    .reset_index(
        name="RECENT_LATE_PAYMENT_COUNT"
    )
)

application_train = (

    application_train.merge(

        recent_late_count,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "RECENT_LATE_PAYMENT_COUNT"
] = (

    application_train[
        "RECENT_LATE_PAYMENT_COUNT"
    ]
    .fillna(0)
)

# WHY THIS IS VERY IMPORTANT ⭐⭐⭐
# This captures:
# active repayment deterioration.
# VERY important enterprise signal.

In [ ]:
# 3. EMPLOYMENT STABILITY FEATURES ⭐⭐⭐
# VERY important in banking.
# YEARS EMPLOYED ⭐⭐⭐

# ------------------------------------------------------------
# YEARS_EMPLOYED
# ------------------------------------------------------------
# Measures total employment duration
# in years for each customer.
#
# Business Meaning:
# Represents employment stability
# and long-term income consistency.
#
# Customers with longer employment history
# may demonstrate:
# - more stable income
# - lower employment volatility
# - stronger financial resilience
# - improved repayment reliability
#
# Formula:
# Absolute Employment Days / 365
#
# Enterprise Use:
# Used in:
# - underwriting assessment
# - affordability analysis
# - customer stability scoring
# - income reliability modeling
# - probability of default estimation
#
# Higher values may indicate:
# - stable professional history
# - stronger repayment capacity
# - lower operational income risk
#
# Lower values may indicate:
# - unstable employment
# - income uncertainty
# - elevated financial vulnerability
#
# NOTE:
# DAYS_EMPLOYED is stored as negative values
# in the original dataset because it represents
# historical days before current application date.
#
# Absolute values convert employment duration
# into interpretable positive years.


# ------------------------------------------------------------
# EMPLOYMENT_TO_AGE_RATIO
# ------------------------------------------------------------
# Measures employment duration relative
# to customer lifetime age.
#
# Business Meaning:
# Estimates career stability and workforce
# participation consistency throughout
# the customer's lifetime.
#
# Formula:
# Employment Duration
# /
# Customer Age
#
# Enterprise Interpretation:
#
# Higher ratios may indicate:
# - long-term workforce consistency
# - stable career progression
# - stronger income continuity
# - mature financial behavior
#
# Lower ratios may indicate:
# - fragmented employment history
# - inconsistent workforce participation
# - income instability
# - elevated affordability uncertainty
#
# Example:
#
# Customer A:
# age = 40 years
# employed = 20 years
#
# ratio = 0.50
#
# Customer B:
# age = 40 years
# employed = 2 years
#
# ratio = 0.05
#
# Customer A demonstrates significantly
# stronger long-term employment stability.
#
# Enterprise Use:
# Used in:
# - customer resilience modeling
# - employment stability assessment
# - affordability reliability analysis
# - behavioral underwriting systems
#
# This feature provides richer behavioral
# context than raw employment duration alone
# because it normalizes employment history
# relative to customer lifecycle stage.
application_train[
    "YEARS_EMPLOYED"
] = (

    abs(
        application_train[
            "DAYS_EMPLOYED"
        ]
    )

    / 365
)

application_train[
    "EMPLOYMENT_TO_AGE_RATIO"
] = (

    abs(
        application_train[
            "DAYS_EMPLOYED"
        ]
    )

    /

    abs(
        application_train[
            "DAYS_BIRTH"
        ]
    )
)



/tmp/ipykernel_1520/4056969649.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[
/tmp/ipykernel_1520/4056969649.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [ ]:
# 4. COMPOSITE BEHAVIORAL RISK SCORE ⭐⭐⭐
# THIS becomes VERY enterprise.
# BUILD NORMALIZED FLAGS ⭐⭐⭐

# ------------------------------------------------------------
# HIGH_DTI_FLAG
# ------------------------------------------------------------
# Binary behavioral risk indicator identifying
# customers with elevated debt burden relative
# to income capacity.
#
# Logic:
# Flag = 1
# if Debt-to-Income Ratio > 1
#
# Business Meaning:
# Customers whose external debt exceeds
# annual income may demonstrate:
#
# - affordability stress
# - over-leverage
# - elevated repayment pressure
# - reduced financial flexibility
#
# Enterprise Use:
# Used in:
# - underwriting rules
# - affordability monitoring
# - leverage risk segmentation
# - behavioral scorecards
#
# This converts continuous financial exposure
# into an interpretable enterprise risk signal.


# ------------------------------------------------------------
# HIGH_LATE_PAYMENT_FLAG
# ------------------------------------------------------------
# Binary behavioral risk indicator identifying
# customers with repeated repayment delays.
#
# Logic:
# Flag = 1
# if Late Payment Count > 5
#
# Business Meaning:
# Customers with frequent missed payments
# may demonstrate:
#
# - chronic repayment instability
# - deteriorating financial behavior
# - recurring liquidity stress
# - elevated delinquency risk
#
# Enterprise Use:
# Used in:
# - early warning systems
# - collections prioritization
# - behavioral monitoring
# - repayment stability analysis
#
# This feature transforms historical repayment
# behavior into a simple operational risk signal
# suitable for explainable enterprise scoring.


# ------------------------------------------------------------
# HIGH_BORROWING_FLAG
# ------------------------------------------------------------
# Binary behavioral indicator identifying
# customers with elevated historical
# borrowing activity.
#
# Logic:
# Flag = 1
# if Previous Application Count > 10
#
# Business Meaning:
# High historical application frequency
# may indicate:
#
# - dependency on external financing
# - aggressive borrowing behavior
# - elevated credit demand intensity
# - potential affordability pressure
#
# Enterprise Use:
# Used in:
# - behavioral credit analysis
# - borrowing intensity monitoring
# - customer dependency assessment
# - credit appetite segmentation
#
# This feature helps identify customers
# exhibiting unusually high loan application
# activity relative to broader portfolio behavior.


# ------------------------------------------------------------
# COMPOSITE BEHAVIORAL RISK LOGIC
# ------------------------------------------------------------
# These engineered binary flags are designed
# to simplify complex behavioral patterns into
# interpretable enterprise risk indicators.
#
# Individually:
#
# - HIGH_DTI_FLAG
#   captures leverage stress
#
# - HIGH_LATE_PAYMENT_FLAG
#   captures repayment instability
#
# - HIGH_BORROWING_FLAG
#   captures aggressive financing behavior
#
# Together:
# these features form the foundation for
# composite behavioral risk scoring systems
# commonly used in enterprise banking AI.
#
# This approach mirrors real-world banking
# systems where multiple weak-to-moderate
# behavioral signals are combined into
# explainable operational risk frameworks.


application_train[
    "HIGH_DTI_FLAG"
] = (

    application_train[
        "DEBT_TO_INCOME"
    ] > 1
).astype(int)

application_train[
    "HIGH_LATE_PAYMENT_FLAG"
] = (

    application_train[
        "LATE_PAYMENT_COUNT"
    ] > 5
).astype(int)

application_train[
    "HIGH_BORROWING_FLAG"
] = (

    application_train[
        "PREVIOUS_APPLICATION_COUNT"
    ] > 10
).astype(int)



/tmp/ipykernel_1520/1791381166.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[
/tmp/ipykernel_1520/1791381166.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[
/tmp/ipykernel_1520/1791381166.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_

In [26]:
# 3. EMPLOYMENT STABILITY FEATURES ⭐⭐⭐

# VERY important in banking.

# YEARS EMPLOYED ⭐⭐⭐

# ------------------------------------------------------------
# YEARS_EMPLOYED
# ------------------------------------------------------------
# Measures total customer employment duration
# in years.
#
# Business Meaning:
# Represents long-term employment stability
# and consistency of income generation.
#
# Customers with longer employment history
# may demonstrate:
#
# - stable professional background
# - predictable income streams
# - lower income volatility
# - improved repayment reliability
#
# Formula:
# Absolute Employment Days / 365
#
# Enterprise Use:
# Used in:
# - underwriting analysis
# - affordability assessment
# - income stability modeling
# - repayment resilience scoring
# - probability of default estimation
#
# Higher values may indicate:
# - stronger financial stability
# - mature employment history
# - reduced operational income risk
#
# Lower values may indicate:
# - unstable employment
# - inconsistent workforce participation
# - elevated affordability uncertainty
#
# NOTE:
# DAYS_EMPLOYED values are stored as
# negative historical offsets relative
# to application date.
#
# Absolute values convert the feature
# into interpretable employment duration.


# ------------------------------------------------------------
# EMPLOYMENT_TO_AGE_RATIO
# ------------------------------------------------------------
# Measures employment duration relative
# to total customer lifetime age.
#
# Business Meaning:
# Estimates long-term workforce participation
# consistency throughout the customer's life.
#
# Formula:
# Employment Duration
# /
# Customer Age
#
# Enterprise Interpretation:
#
# Higher ratios may indicate:
#
# - sustained workforce participation
# - stable career progression
# - stronger long-term income continuity
# - mature financial behavior
#
# Lower ratios may indicate:
#
# - fragmented employment history
# - employment instability
# - inconsistent income generation
# - elevated repayment uncertainty
#
# Example:
#
# Customer A:
# age = 40 years
# employed = 20 years
#
# ratio = 0.50
#
# Customer B:
# age = 40 years
# employed = 2 years
#
# ratio = 0.05
#
# Customer A demonstrates significantly
# stronger employment consistency and
# potentially more reliable repayment capacity.
#
# Enterprise Use:
# Used in:
# - customer resilience modeling
# - affordability reliability analysis
# - employment risk segmentation
# - behavioral underwriting systems
#
# This feature provides richer behavioral
# context than raw employment duration alone
# because it normalizes employment history
# relative to customer lifecycle stage.
#
# Enterprise Insight:
# Customers with stable long-term employment
# histories often exhibit:
#
# - more predictable income behavior
# - lower financial volatility
# - improved repayment consistency
# - reduced probability of default
application_train[
    "YEARS_EMPLOYED"
] = (

    abs(
        application_train[
            "DAYS_EMPLOYED"
        ]
    )

    / 365
)

application_train[
    "EMPLOYMENT_TO_AGE_RATIO"
] = (

    abs(
        application_train[
            "DAYS_EMPLOYED"
        ]
    )

    /

    abs(
        application_train[
            "DAYS_BIRTH"
        ]
    )
)



In [27]:
# 4. COMPOSITE BEHAVIORAL RISK SCORE ⭐⭐⭐

# THIS becomes VERY enterprise.

# BUILD NORMALIZED FLAGS ⭐⭐⭐

# ------------------------------------------------------------
# HIGH_DTI_FLAG
# ------------------------------------------------------------
# Binary behavioral risk indicator identifying
# customers with elevated leverage pressure
# relative to income capacity.
#
# Logic:
# Flag = 1
# if Debt-to-Income Ratio > 1
#
# Business Meaning:
# Customers whose total external debt exceeds
# annual income may demonstrate:
#
# - affordability stress
# - excessive leverage exposure
# - reduced repayment flexibility
# - elevated financial vulnerability
#
# Enterprise Use:
# Used in:
# - affordability assessment
# - leverage risk monitoring
# - underwriting policies
# - explainable credit scoring systems
#
# This feature converts continuous debt burden
# into an interpretable operational risk signal.


# ------------------------------------------------------------
# HIGH_LATE_PAYMENT_FLAG
# ------------------------------------------------------------
# Binary behavioral risk indicator identifying
# customers with repeated repayment delays.
#
# Logic:
# Flag = 1
# if Late Payment Count > 5
#
# Business Meaning:
# Repeated missed payments may indicate:
#
# - chronic repayment instability
# - deteriorating financial behavior
# - ongoing liquidity stress
# - elevated delinquency vulnerability
#
# Enterprise Use:
# Used in:
# - behavioral monitoring
# - collections prioritization
# - early warning systems
# - repayment stability assessment
#
# This feature simplifies historical repayment
# behavior into a highly interpretable
# operational delinquency signal.


# ------------------------------------------------------------
# HIGH_BORROWING_FLAG
# ------------------------------------------------------------
# Binary behavioral indicator identifying
# customers with elevated historical
# borrowing intensity.
#
# Logic:
# Flag = 1
# if Previous Application Count > 10
#
# Business Meaning:
# High historical application frequency
# may indicate:
#
# - dependency on external financing
# - aggressive borrowing behavior
# - elevated credit demand intensity
# - affordability pressure
#
# Enterprise Use:
# Used in:
# - customer borrowing analysis
# - financing dependency assessment
# - behavioral segmentation
# - credit appetite monitoring
#
# This feature identifies customers
# exhibiting unusually high borrowing
# activity relative to broader portfolio behavior.


# ------------------------------------------------------------
# BEHAVIORAL_RISK_SCORE
# ------------------------------------------------------------
# Composite behavioral risk score aggregating
# multiple independent financial stress signals
# into a single interpretable enterprise metric.
#
# Formula:
#
# HIGH_DTI_FLAG
# +
# HIGH_LATE_PAYMENT_FLAG
# +
# HIGH_BORROWING_FLAG
#
# Score Range:
#
# 0 -> low observed behavioral risk
# 1 -> isolated elevated risk signal
# 2 -> multiple concurrent risk indicators
# 3 -> severe combined behavioral risk
#
# Business Meaning:
# This feature models cumulative behavioral
# financial stress across multiple dimensions:
#
# - leverage pressure
# - repayment instability
# - borrowing aggressiveness
#
# Enterprise Interpretation:
#
# Higher scores may indicate:
#
# - compound affordability deterioration
# - elevated repayment vulnerability
# - chronic financial stress
# - higher probability of default
#
# Lower scores may indicate:
#
# - stable borrowing behavior
# - manageable leverage exposure
# - consistent repayment discipline
#
# Enterprise Use:
# Used in:
#
# - behavioral scorecards
# - explainable AI systems
# - early warning frameworks
# - customer risk segmentation
# - underwriting intelligence platforms
#
# Enterprise Insight:
# Modern banking systems rarely rely on
# single isolated variables.
#
# Instead, enterprise credit risk systems
# combine multiple weak-to-moderate
# behavioral indicators into composite
# explainable risk intelligence frameworks.
#
# This feature represents a simplified
# version of real-world enterprise
# behavioral risk aggregation systems.
application_train[
    "HIGH_DTI_FLAG"
] = (

    application_train[
        "DEBT_TO_INCOME"
    ] > 1
).astype(int)

application_train[
    "HIGH_LATE_PAYMENT_FLAG"
] = (

    application_train[
        "LATE_PAYMENT_COUNT"
    ] > 5
).astype(int)

application_train[
    "HIGH_BORROWING_FLAG"
] = (

    application_train[
        "PREVIOUS_APPLICATION_COUNT"
    ] > 10
).astype(int)
# COMPOSITE SCORE ⭐⭐⭐
application_train[
    "BEHAVIORAL_RISK_SCORE"
] = (

    application_train[
        "HIGH_DTI_FLAG"
    ]

    +

    application_train[
        "HIGH_LATE_PAYMENT_FLAG"
    ]

    +

    application_train[
        "HIGH_BORROWING_FLAG"
    ]
)

In [25]:
# 5. NONLINEAR RISK BUCKETS ⭐⭐⭐

# VERY important.

# DTI RISK BUCKETS ⭐⭐⭐
application_train[
    "DTI_RISK_BUCKET"
] = pd.cut(

    application_train[
        "DEBT_TO_INCOME"
    ],

    bins=[-1, 0.5, 1, 2, 100],

    labels=[
        "LOW",
        "MEDIUM",
        "HIGH",
        "SEVERE"
    ]
)

/tmp/ipykernel_1520/3117679562.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [ ]:
# ============================================================
# SAVE ENGINEERED DATASET
# ============================================================

application_train.to_csv(

    "../data/processed/engineered_credit_risk_dataset.csv",

    index=False
)

print(
    "Engineered dataset saved successfully."
)

Engineered dataset saved successfully.


In [ ]:
# | Category             | Feature                    | Business Meaning                                                             |
# | -------------------- | -------------------------- | ---------------------------------------------------------------------------- |
# | Affordability        | ANNUITY_TO_INCOME          | Measures repayment burden relative to customer income capacity               |
# | Affordability        | CREDIT_TO_INCOME           | Measures requested credit exposure relative to income                        |
# | Affordability        | FREE_CASH_FLOW             | Estimates remaining customer income after repayment obligations              |
# | Leverage             | TOTAL_EXTERNAL_DEBT        | Total external debt exposure across bureau credit relationships              |
# | Leverage             | DEBT_TO_INCOME             | Measures debt pressure relative to customer income                           |
# | Exposure             | BUREAU_RECORD_COUNT        | Number of external credit relationships linked to customer                   |
# | Exposure             | DEBT_PER_BUREAU_RECORD     | Measures debt concentration across credit relationships                      |
# | Borrowing Behavior   | PREVIOUS_APPLICATION_COUNT | Measures historical loan application frequency                               |
# | Borrowing Behavior   | APPLICATIONS_PER_INCOME    | Measures borrowing aggressiveness relative to income                         |
# | Repayment Behavior   | LATE_PAYMENT_COUNT         | Number of historically late installment payments                             |
# | Repayment Behavior   | MISSED_PAYMENTS_PER_LOAN   | Measures repayment instability across historical loans                       |
# | Repayment Severity   | MAX_CREDIT_DAYS_OVERDUE    | Maximum historical delinquency severity observed                             |
# | Repayment Severity   | OVERDUE_PER_BUREAU_RECORD  | Measures delinquency severity relative to credit ecosystem size              |
# | Repayment Stability  | REPAYMENT_STABILITY        | Normalized repayment consistency score                                       |
# | Payment Punctuality  | AVG_PAYMENT_DELAY          | Average lateness severity across installment payments                        |
# | Temporal Risk        | RECENT_LATE_PAYMENT_COUNT  | Measures recent repayment deterioration within recent payment window         |
# | Employment Stability | YEARS_EMPLOYED             | Measures employment duration and income consistency                          |
# | Employment Stability | EMPLOYMENT_TO_AGE_RATIO    | Measures long-term workforce participation stability                         |
# | Risk Flags           | HIGH_DTI_FLAG              | Indicates elevated leverage pressure                                         |
# | Risk Flags           | HIGH_LATE_PAYMENT_FLAG     | Indicates repeated repayment instability                                     |
# | Risk Flags           | HIGH_BORROWING_FLAG        | Indicates aggressive historical borrowing behavior                           |
# | Composite Risk       | BEHAVIORAL_RISK_SCORE      | Aggregated behavioral financial stress score combining multiple risk signals |


In [ ]:
# | Advanced Feature              | Meaning                          |
# | ----------------------------- | -------------------------------- |
# | last 3 applications           | recent borrowing behavior        |
# | last 60/90/180 days           | recent deterioration             |
# | lag features                  | behavioral trajectory            |
# | weighted means                | temporal importance              |
# | installment sequence analysis | payment evolution                |
# | recent delinquency windows    | dynamic stress                   |
# | active loan recency           | ongoing exposure                 |
# | neighbor target means         | customer similarity intelligence |


In [29]:
# LATE PAYMENTS LAST 90 DAYS ⭐⭐⭐

# This becomes:

# active deterioration signal.

# ============================================================
# RECENT 90-DAY PAYMENT WINDOW
# ============================================================

recent_90d_payments = (

    installments_payments[

        installments_payments[
            "DAYS_INSTALMENT"
        ] > -90
    ]
)

# ============================================================
# RECENT LATE PAYMENTS
# ============================================================

recent_90d_payments[
    "RECENT_LATE_FLAG"
] = (

    recent_90d_payments[
        "PAYMENT_DELAY"
    ] > 0
).astype(int)

recent_90d_late_count = (

    recent_90d_payments.groupby(
        "SK_ID_CURR"
    )[
        "RECENT_LATE_FLAG"
    ]
    .sum()
    .reset_index()
)

recent_90d_late_count.columns = [

    "SK_ID_CURR",

    "LATE_PAYMENTS_LAST_90D"
]

application_train = (

    application_train.merge(

        recent_90d_late_count,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "LATE_PAYMENTS_LAST_90D"
] = (

    application_train[
        "LATE_PAYMENTS_LAST_90D"
    ]
    .fillna(0)
)

In [30]:


# ============================================================
# RECENT AVG PAYMENT DELAY
# ============================================================

recent_avg_delay = (

    recent_90d_payments.groupby(
        "SK_ID_CURR"
    )[
        "PAYMENT_DELAY"
    ]
    .mean()
    .reset_index()
)

recent_avg_delay.columns = [

    "SK_ID_CURR",

    "AVG_PAYMENT_DELAY_LAST_90D"
]

application_train = (

    application_train.merge(

                recent_avg_delay,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "AVG_PAYMENT_DELAY_LAST_90D"
] = (

    application_train[
        "AVG_PAYMENT_DELAY_LAST_90D"
    ]
    .fillna(0)
)

In [32]:
# RECENT VS HISTORICAL DELAY RATIO ⭐⭐⭐

# THIS becomes:

# deterioration trend intelligence.
# ============================================================
# RECENT VS HISTORICAL PAYMENT STRESS
# ============================================================

application_train[
    "RECENT_TO_HISTORICAL_DELAY_RATIO"
] = (

    application_train[
        "AVG_PAYMENT_DELAY_LAST_90D"
    ]

    /

    (
        application_train[
            "AVG_PAYMENT_DELAY"
        ]
        +
        1
    )
)

/tmp/ipykernel_1520/1800063654.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [ ]:
# RECENT BORROWING ACTIVITY ⭐⭐⭐

# THIS becomes:

# borrowing acceleration intelligence.
# ============================================================
# RECENT APPLICATION ACTIVITY
# ============================================================

recent_previous_apps = (

    previous_application[

        previous_application[
            "DAYS_DECISION"
        ] > -180
    ]
)

recent_application_count = (

    recent_previous_apps.groupby(
        "SK_ID_CURR"
    )
    .size()
    .reset_index(
        name="RECENT_APPLICATION_COUNT"
    )
)

application_train = (

    application_train.merge(

        recent_application_count,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "RECENT_APPLICATION_COUNT"
] = (

    application_train[
        "RECENT_APPLICATION_COUNT"
    ]
    .fillna(0)
)

In [34]:
# BORROWING ACCELERATION RATIO ⭐⭐⭐

# VERY enterprise feature.

# ============================================================
# BORROWING ACCELERATION
# ============================================================

application_train[
    "BORROWING_ACCELERATION_RATIO"
] = (

    application_train[
        "RECENT_APPLICATION_COUNT"
    ]

    /

    (
        application_train[
            "PREVIOUS_APPLICATION_COUNT"
        ]
        +
        1
    )
)

/tmp/ipykernel_1520/3562102715.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [35]:
# RECENT PAYMENT STABILITY SCORE ⭐⭐⭐

# THIS becomes:

# short-term behavioral resilience.

# ============================================================
# RECENT PAYMENT STABILITY
# ============================================================

application_train[
    "RECENT_PAYMENT_STABILITY"
] = (

    1

    /

    (
        1
        +
        application_train[
            "LATE_PAYMENTS_LAST_90D"
        ]
    )
)

/tmp/ipykernel_1520/1524361987.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [37]:
# ============================================================
# ADVANCED ENTERPRISE FEATURE ENGINEERING ⭐⭐⭐
# ============================================================
# PURPOSE:
# Add stronger behavioral,
# affordability,
# repayment trajectory,
# and temporal intelligence features.
#
# Inspired by:
# - Kaggle winning solutions
# - enterprise banking methodologies
# - behavioral credit risk modeling
# ============================================================


# ============================================================
# 1. CREDIT TO ANNUITY RATIO ⭐⭐⭐
# ============================================================
# Measures:
# loan size relative to repayment burden.

application_train[
    "CREDIT_TO_ANNUITY_RATIO"
] = (

    application_train[
        "AMT_CREDIT"
    ]

    /

    (
        application_train[
            "AMT_ANNUITY"
        ]
        + 1
    )
)


# ============================================================
# 2. CREDIT TO GOODS PRICE RATIO ⭐⭐⭐
# ============================================================
# Measures:
# financing proportion relative to goods value.
#
# Higher values may indicate:
# lower customer contribution.

application_train[
    "CREDIT_TO_GOODS_RATIO"
] = (

    application_train[
        "AMT_CREDIT"
    ]

    /

    (
        application_train[
            "AMT_GOODS_PRICE"
        ]
        + 1
    )
)


# ============================================================
# 3. DOWN PAYMENT FEATURE ⭐⭐⭐
# ============================================================
# Measures:
# customer upfront contribution.

application_train[
    "DOWN_PAYMENT"
] = (

    application_train[
        "AMT_GOODS_PRICE"
    ]

    -

    application_train[
        "AMT_CREDIT"
    ]
)


# ============================================================
# 4. PAYMENT DEFICIT ⭐⭐⭐
# ============================================================
# Measures:
# whether customer underpaid installments.

installments_payments[
    "PAYMENT_DEFICIT"
] = (

    installments_payments[
        "AMT_PAYMENT"
    ]

    -

    installments_payments[
        "AMT_INSTALMENT"
    ]
)


# ============================================================
# 5. AVG PAYMENT DEFICIT ⭐⭐⭐
# ============================================================
# Customer-level repayment sufficiency.

customer_avg_payment_deficit = (

    installments_payments.groupby(
        "SK_ID_CURR"
    )[
        "PAYMENT_DEFICIT"
    ]
    .mean()
    .reset_index()
)

customer_avg_payment_deficit.columns = [

    "SK_ID_CURR",

    "AVG_PAYMENT_DEFICIT"
]

application_train = (

    application_train.merge(

        customer_avg_payment_deficit,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "AVG_PAYMENT_DEFICIT"
] = (

    application_train[
        "AVG_PAYMENT_DEFICIT"
    ]
    .fillna(0)
)


# ============================================================
# 6. RECENT PAYMENT DEFICIT ⭐⭐⭐
# ============================================================
# Measures:
# recent repayment deterioration.

recent_installments = (

    installments_payments[

        installments_payments[
            "DAYS_INSTALMENT"
        ] > -90
    ]
)

recent_payment_deficit = (

    recent_installments.groupby(
        "SK_ID_CURR"
    )[
        "PAYMENT_DEFICIT"
    ]
    .mean()
    .reset_index()
)

recent_payment_deficit.columns = [

    "SK_ID_CURR",

    "RECENT_PAYMENT_DEFICIT"
]

application_train = (

    application_train.merge(

        recent_payment_deficit,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "RECENT_PAYMENT_DEFICIT"
] = (

    application_train[
        "RECENT_PAYMENT_DEFICIT"
    ]
    .fillna(0)
)


# ============================================================
# 7. RECENT VS HISTORICAL PAYMENT DEFICIT ⭐⭐⭐
# ============================================================
# Measures:
# repayment deterioration trajectory.

application_train[
    "PAYMENT_DEFICIT_TREND"
] = (

    application_train[
        "RECENT_PAYMENT_DEFICIT"
    ]

    /

    (
        application_train[
            "AVG_PAYMENT_DEFICIT"
        ]
        + 1
    )
)


# ============================================================
# 8. CUSTOMER AGE IN YEARS ⭐⭐⭐
# ============================================================

application_train[
    "AGE_YEARS"
] = (

    abs(
        application_train[
            "DAYS_BIRTH"
        ]
    )

    / 365
)


# ============================================================
# 9. AGE GROUP FEATURE ⭐⭐⭐
# ============================================================
# Similar concept to winning AGE_INT feature.

application_train[
    "AGE_GROUP"
] = pd.cut(

    application_train[
        "AGE_YEARS"
    ],

    bins=[18,25,35,45,55,65,100],

    labels=[
        1,2,3,4,5,6
    ]
)


# ============================================================
# 10. MEAN BUREAU CREDIT DAYS ⭐⭐⭐
# ============================================================
# Measures:
# average external credit history recency.

bureau_credit_days_mean = (

    bureau.groupby(
        "SK_ID_CURR"
    )[
        "DAYS_CREDIT"
    ]
    .mean()
    .reset_index()
)

bureau_credit_days_mean.columns = [

    "SK_ID_CURR",

    "MEAN_DAYS_CREDIT"
]

application_train = (

    application_train.merge(

        bureau_credit_days_mean,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "MEAN_DAYS_CREDIT"
] = (

    application_train[
        "MEAN_DAYS_CREDIT"
    ]
    .fillna(0)
)


# ============================================================
# 11. ACTIVE CREDIT RECENCY ⭐⭐⭐
# ============================================================
# Measures:
# recency of active external loans.

active_bureau = (

    bureau[
        bureau[
            "CREDIT_ACTIVE"
        ] == "Active"
    ]
)

recent_active_credit = (

    active_bureau.groupby(
        "SK_ID_CURR"
    )[
        "DAYS_CREDIT"
    ]
    .max()
    .reset_index()
)

recent_active_credit.columns = [

    "SK_ID_CURR",

    "LAST_ACTIVE_DAYS_CREDIT"
]

application_train = (

    application_train.merge(

        recent_active_credit,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "LAST_ACTIVE_DAYS_CREDIT"
] = (

    application_train[
        "LAST_ACTIVE_DAYS_CREDIT"
    ]
    .fillna(0)
)


# ============================================================
# 12. ACTIVE DEBT RATIO ⭐⭐⭐
# ============================================================
# Measures:
# proportion of debt currently active.

active_debt = (

    active_bureau.groupby(
        "SK_ID_CURR"
    )[
        "AMT_CREDIT_SUM_DEBT"
    ]
    .sum()
    .reset_index()
)

active_debt.columns = [

    "SK_ID_CURR",

    "ACTIVE_EXTERNAL_DEBT"
]

application_train = (

    application_train.merge(

        active_debt,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "ACTIVE_EXTERNAL_DEBT"
] = (

    application_train[
        "ACTIVE_EXTERNAL_DEBT"
    ]
    .fillna(0)
)

application_train[
    "ACTIVE_DEBT_RATIO"
] = (

    application_train[
        "ACTIVE_EXTERNAL_DEBT"
    ]

    /

    (
        application_train[
            "TOTAL_EXTERNAL_DEBT"
        ]
        + 1
    )
)


# ============================================================
# 13. MAX INSTALLMENT FEATURE ⭐⭐⭐
# ============================================================
# Measures:
# maximum historical installment obligation.

max_installment = (

    installments_payments.groupby(
        "SK_ID_CURR"
    )[
        "AMT_INSTALMENT"
    ]
    .max()
    .reset_index()
)

max_installment.columns = [

    "SK_ID_CURR",

    "MAX_INSTALLMENT"
]

application_train = (

    application_train.merge(

        max_installment,

        on="SK_ID_CURR",

        how="left"
    )
)

application_train[
    "MAX_INSTALLMENT"
] = (

    application_train[
        "MAX_INSTALLMENT"
    ]
    .fillna(0)
)


# ============================================================
# 14. ANNUITY TO MAX INSTALLMENT RATIO ⭐⭐⭐
# ============================================================
# Measures:
# current repayment burden relative
# to historical repayment capacity.

application_train[
    "ANNUITY_TO_MAX_INSTALLMENT_RATIO"
] = (

    application_train[
        "AMT_ANNUITY"
    ]

    /

    (
        application_train[
            "MAX_INSTALLMENT"
        ]
        + 1
    )
)


# ============================================================
# 15. RECENT PAYMENT DELAY TREND ⭐⭐⭐
# ============================================================
# Measures:
# whether repayment behavior
# is worsening recently.

application_train[
    "RECENT_DELAY_TREND"
] = (

    application_train[
        "AVG_PAYMENT_DELAY_LAST_90D"
    ]

    -

    application_train[
        "AVG_PAYMENT_DELAY"
    ]
)


# ============================================================
# FINAL VALIDATION
# ============================================================

print(
    "Advanced enterprise features generated successfully."
)

print(application_train.shape)

/tmp/ipykernel_1520/1021384151.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[
/tmp/ipykernel_1520/1021384151.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[
/tmp/ipykernel_1520/1021384151.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application

Advanced enterprise features generated successfully.
(307511, 166)


/tmp/ipykernel_1520/1021384151.py:503: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[
/tmp/ipykernel_1520/1021384151.py:529: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[


In [ ]:
| Category                   | Feature                          | Formula / Logic                                         | Business Meaning                                                  |
| -------------------------- | -------------------------------- | ------------------------------------------------------- | ----------------------------------------------------------------- |
| Affordability              | ANNUITY_TO_INCOME                | `AMT_ANNUITY / AMT_INCOME_TOTAL`                        | Measures repayment burden relative to customer income             |
| Affordability              | CREDIT_TO_INCOME                 | `AMT_CREDIT / AMT_INCOME_TOTAL`                         | Measures loan exposure relative to income capacity                |
| Affordability              | FREE_CASH_FLOW                   | `AMT_INCOME_TOTAL - AMT_ANNUITY`                        | Estimates remaining disposable income after loan obligations      |
| Affordability ⭐            | CREDIT_TO_ANNUITY_RATIO          | `AMT_CREDIT / AMT_ANNUITY`                              | Measures loan size relative to repayment burden                   |
| Financing Structure ⭐      | CREDIT_TO_GOODS_RATIO            | `AMT_CREDIT / AMT_GOODS_PRICE`                          | Measures financing proportion relative to purchased asset value   |
| Financing Structure ⭐      | DOWN_PAYMENT                     | `AMT_GOODS_PRICE - AMT_CREDIT`                          | Estimates customer upfront contribution and financial commitment  |
| Leverage                   | TOTAL_EXTERNAL_DEBT              | Sum of bureau debt exposure                             | Total external debt obligations across institutions               |
| Leverage                   | DEBT_TO_INCOME                   | `TOTAL_EXTERNAL_DEBT / AMT_INCOME_TOTAL`                | Measures leverage pressure relative to income                     |
| Exposure                   | BUREAU_RECORD_COUNT              | Count of bureau credit relationships                    | Measures external credit ecosystem size                           |
| Exposure                   | DEBT_PER_BUREAU_RECORD           | `TOTAL_EXTERNAL_DEBT / BUREAU_RECORD_COUNT`             | Measures debt concentration across relationships                  |
| Active Exposure ⭐          | ACTIVE_EXTERNAL_DEBT             | Sum of active bureau debt                               | Measures currently active external obligations                    |
| Active Exposure ⭐          | ACTIVE_DEBT_RATIO                | `ACTIVE_EXTERNAL_DEBT / TOTAL_EXTERNAL_DEBT`            | Measures proportion of debt currently active                      |
| Bureau Recency ⭐           | MEAN_DAYS_CREDIT                 | Mean bureau DAYS_CREDIT                                 | Measures average recency of historical external credit            |
| Bureau Recency ⭐           | LAST_ACTIVE_DAYS_CREDIT          | Max DAYS_CREDIT for active loans                        | Measures most recent active external credit relationship          |
| Borrowing Behavior         | PREVIOUS_APPLICATION_COUNT       | Count of historical applications                        | Measures historical borrowing intensity                           |
| Borrowing Behavior         | APPLICATIONS_PER_INCOME          | `PREVIOUS_APPLICATION_COUNT / AMT_INCOME_TOTAL`         | Measures borrowing aggressiveness relative to income              |
| Borrowing Velocity ⭐       | RECENT_APPLICATION_COUNT         | Count of applications within recent 180 days            | Measures recent financing demand intensity                        |
| Borrowing Velocity ⭐       | BORROWING_ACCELERATION_RATIO     | `RECENT_APPLICATION_COUNT / PREVIOUS_APPLICATION_COUNT` | Measures acceleration of borrowing behavior                       |
| Repayment Behavior         | LATE_PAYMENT_COUNT               | Count of delayed installment payments                   | Measures historical repayment instability                         |
| Repayment Behavior         | MISSED_PAYMENTS_PER_LOAN         | `LATE_PAYMENT_COUNT / PREVIOUS_APPLICATION_COUNT`       | Measures repayment instability normalized by loan activity        |
| Repayment Severity         | AVG_PAYMENT_DELAY                | Mean payment delay days                                 | Measures average delinquency severity                             |
| Temporal Delinquency ⭐     | LATE_PAYMENTS_LAST_90D           | Count of late payments within recent 90 days            | Measures active repayment deterioration                           |
| Temporal Delinquency ⭐     | AVG_PAYMENT_DELAY_LAST_90D       | Mean delay within recent 90 days                        | Measures recent delinquency severity                              |
| Behavioral Trajectory ⭐    | RECENT_TO_HISTORICAL_DELAY_RATIO | `AVG_PAYMENT_DELAY_LAST_90D / AVG_PAYMENT_DELAY`        | Measures worsening vs improving repayment behavior                |
| Behavioral Trend ⭐         | RECENT_DELAY_TREND               | `AVG_PAYMENT_DELAY_LAST_90D - AVG_PAYMENT_DELAY`        | Measures recent repayment deterioration trend                     |
| Payment Sufficiency ⭐      | PAYMENT_DEFICIT                  | `AMT_PAYMENT - AMT_INSTALMENT`                          | Measures underpayment vs installment obligation                   |
| Payment Sufficiency ⭐      | AVG_PAYMENT_DEFICIT              | Mean PAYMENT_DEFICIT                                    | Measures long-term repayment sufficiency                          |
| Payment Sufficiency ⭐      | RECENT_PAYMENT_DEFICIT           | Mean PAYMENT_DEFICIT in recent 90 days                  | Measures recent repayment sufficiency deterioration               |
| Behavioral Deterioration ⭐ | PAYMENT_DEFICIT_TREND            | `RECENT_PAYMENT_DEFICIT / AVG_PAYMENT_DEFICIT`          | Measures worsening repayment sufficiency trend                    |
| Delinquency Severity       | MAX_CREDIT_DAYS_OVERDUE          | Maximum bureau overdue days                             | Measures worst historical delinquency severity                    |
| Delinquency Severity       | OVERDUE_PER_BUREAU_RECORD        | `MAX_CREDIT_DAYS_OVERDUE / BUREAU_RECORD_COUNT`         | Measures delinquency severity normalized by exposure size         |
| Repayment Stability        | REPAYMENT_STABILITY              | `1 / (1 + LATE_PAYMENT_COUNT)`                          | Measures long-term repayment consistency                          |
| Temporal Stability ⭐       | RECENT_PAYMENT_STABILITY         | `1 / (1 + LATE_PAYMENTS_LAST_90D)`                      | Measures short-term repayment consistency                         |
| Historical Capacity ⭐      | MAX_INSTALLMENT                  | Maximum historical installment payment                  | Measures peak historical repayment capacity                       |
| Historical Capacity ⭐      | ANNUITY_TO_MAX_INSTALLMENT_RATIO | `AMT_ANNUITY / MAX_INSTALLMENT`                         | Measures current repayment burden relative to historical capacity |
| Employment Stability       | YEARS_EMPLOYED                   | `abs(DAYS_EMPLOYED) / 365`                              | Measures employment duration and income consistency               |
| Employment Stability       | EMPLOYMENT_TO_AGE_RATIO          | `DAYS_EMPLOYED / DAYS_BIRTH`                            | Measures workforce participation consistency                      |
| Demographics ⭐             | AGE_YEARS                        | `abs(DAYS_BIRTH) / 365`                                 | Customer age in years                                             |
| Demographics ⭐             | AGE_GROUP                        | Age bucket segmentation                                 | Captures lifecycle-stage behavioral patterns                      |
| Composite Risk             | HIGH_DTI_FLAG                    | `DEBT_TO_INCOME > 1`                                    | Binary leverage stress indicator                                  |
| Composite Risk             | HIGH_LATE_PAYMENT_FLAG           | `LATE_PAYMENT_COUNT > 5`                                | Binary repayment instability indicator                            |
| Composite Risk             | HIGH_BORROWING_FLAG              | `PREVIOUS_APPLICATION_COUNT > 10`                       | Binary aggressive borrowing indicator                             |
| Composite Risk             | BEHAVIORAL_RISK_SCORE            | Sum of risk flags                                       | Aggregated behavioral financial stress score                      |


In [39]:
# EXT_SOURCE MEAN
application_train[
    "EXT_SOURCE_MEAN"
] = (

    application_train[
        [
            "EXT_SOURCE_1",
            "EXT_SOURCE_2",
            "EXT_SOURCE_3"
        ]
    ]
    .mean(axis=1)
)
# EXT_SOURCE STD ⭐⭐⭐


application_train[
    "EXT_SOURCE_STD"
] = (

    application_train[
        [
            "EXT_SOURCE_1",
            "EXT_SOURCE_2",
            "EXT_SOURCE_3"
        ]
    ]
    .std(axis=1)
)


# EXT_SOURCE × CREDIT RATIO ⭐⭐⭐



application_train[
    "CREDIT_EXT_RATIO"
] = (

    application_train[
        "AMT_CREDIT"
    ]

    /

    (
        application_train[
            "EXT_SOURCE_MEAN"
        ]
        + 0.0001
    )
)

/tmp/ipykernel_1520/3294491937.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[
/tmp/ipykernel_1520/3294491937.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train[
/tmp/ipykernel_1520/3294491937.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_

In [41]:
# ============================================================
# SAVE UPDATED ENGINEERED DATASET
# ============================================================

application_train.to_csv(

    "../data/processed/engineered_credit_risk_dataset.csv",

    index=False
)

print(
    "Updated engineered dataset saved successfully."
)

Updated engineered dataset saved successfully.
